# 05. Głębokie Uczenie Maszynowe (Zabezpieczenie Pełnym MEMMAP)

Kolejne potężne rozwiązanie dla architektury Big Data – **w ogóle nie ładujemy danych do RAMu**.
Notatnik ten korzysta z techniki *Memory-Mappingu*, cofa operacje wczytywania z pamięci konwencjonalnej na fizyczny interfejs dysku twardego bez obciążania pamięci.

Najpierw jednorazowo iterujemy przez wielki `transformed_data_train.csv` (kroki po 100 tysięcy wierszy), dynamicznie ignorując absolutnie wszystkie kolumny kategoryczne lub tekstowe, do surowego formatu ciągłej macierzy C-Array (`.npy`). Omija to błędy napotykane przy niekompatybilnych ciągach znakowych bez narażenia Pytorch na zatrzymanie Kernela.

In [1]:
%load_ext autoreload
%autoreload 2

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import polars as pl
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import gc
import warnings
import joblib
warnings.filterwarnings('ignore')

from models import TimeSeriesLSTM, TimeSeriesTransformer, get_device

SEQ_LEN = 36         
BATCH_SIZE = 1024    
NUM_EPOCHS = 3       
LEARNING_RATE = 1e-3

# ZUNIFIKOWANE ODCINANIE TYPÓW (Aby Test korzystał z tych samych co Train)
preproc_state = joblib.load('preprocessor_fitted.joblib')
FEATURE_COLS = preproc_state.numeric_cols_

## 1. Kompilator Binarny Memory Map (`.npy`)

In [2]:
def create_memmaps(csv_path, x_path, y_path, feature_cols, target_col='x2'):
    if os.path.exists(x_path) and os.path.exists(y_path):
        print(f"Macierze binarne {x_path} i {y_path} już istnieją! Zgłaszam gotowość.\n")
        return
        
    print("1/3 Otwieranie strumieniowe do kalkulacji fizycznych rozmiarów RAM...")
    total_rows = pl.scan_csv(csv_path).select(pl.len()).collect().item()
    
    print(f"2/3 Zrzut {total_rows} rzędów na wirtualnej macierzy dyskowej (.npy)...")
    X_mm = np.memmap(x_path, dtype='float32', mode='w+', shape=(total_rows, len(feature_cols)))
    y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(total_rows,))
    
    print("3/3 Grawerowanie wierszy do bezpiecznych bloków strumieniowych (OOM-proof)...")
    chunksize = 200000
    start_idx = 0
    
    for chunk in tqdm(pd.read_csv(csv_path, chunksize=chunksize, low_memory=False), 
                      total=(total_rows//chunksize) + 1, desc="Konwersja Binarna"):
        
        valid_feats = [c for c in feature_cols if c in chunk.columns]
        assert len(valid_feats) == len(feature_cols), f"Krytyczny błąd: W paczce danych brakuje wymaganych kolumn cech! Znaleziono {len(valid_feats)} z {len(feature_cols)}."
        X_chunk = chunk[valid_feats].fillna(0.0).values.astype(np.float32)
        
        if target_col in chunk.columns:
            y_chunk = chunk[target_col].fillna(0.0).values.astype(np.float32)
        else:
            y_chunk = np.zeros(len(chunk), dtype=np.float32)
        
        end_idx = start_idx + len(chunk)
        X_mm[start_idx:end_idx] = X_chunk
        y_mm[start_idx:end_idx] = y_chunk
        
        start_idx = end_idx
        X_mm.flush() 
        y_mm.flush()
        
    print("Pomyślnie scalono matryce dla wejść Deep Learning !!!")

create_memmaps('data/transformed_data_train.csv', 'data/data_train_X.npy', 'data/data_train_y.npy', FEATURE_COLS)


Macierze binarne data/data_train_X.npy i data/data_train_y.npy już istnieją! Zgłaszam gotowość.



## 2. Dedykowany System DataLoader (Mapowany na I/O)

In [3]:
class MemmapTimeSeriesDataset(Dataset):
    def __init__(self, csv_path, x_path, y_path, feature_cols, seq_len=36, target_col='x2'):
        print("\nPobieranie map stacji urządzeń chroniących przebiegi sekwencyjne...")
        df_devs = pl.read_csv(csv_path, columns=['deviceId'])
        total_rows = len(df_devs)

        # Twarde wyliczanie złączy blokowych 1:1, całkowicie uniezależnione od wersji grupujących w bibliotece
        dev_array = df_devs.to_series().to_numpy()
        changes = np.where(dev_array[:-1] != dev_array[1:])[0] + 1
        device_counts = np.diff(np.concatenate(([0], changes, [total_rows])))
        
        assert len(dev_array) == 0 or len(np.unique(dev_array)) == len(changes) + 1, "Krytyczny błąd: Urządzenia nie tworzą spójnych bloków!"
        
        del df_devs
        gc.collect()
        
        self.feature_cols = feature_cols
        print("Mapowanie wskaźników z dysku do RAM (Zero realnego Loadingu)...")
        self.X = np.memmap(x_path, dtype='float32', mode='r', shape=(total_rows, len(self.feature_cols)))
        self.y = np.memmap(y_path, dtype='float32', mode='r', shape=(total_rows,))
        
        self.seq_len = seq_len
        self.indices = []
        
        current_idx = 0
        for count in tqdm(device_counts, desc="Kompilacja chronologiczna (Indices)"):
            if count >= self.seq_len:
                valid_starts = np.arange(current_idx, current_idx + count - self.seq_len + 1)
                self.indices.extend(valid_starts)
            current_idx += count

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        end = start + self.seq_len
        
        x_seq = torch.tensor(np.array(self.X[start:end]), dtype=torch.float32)
        y_val = torch.tensor(self.y[end - 1], dtype=torch.float32)
        return x_seq, y_val

train_dataset = MemmapTimeSeriesDataset('data/transformed_data_train.csv', 'data/data_train_X.npy', 'data/data_train_y.npy', FEATURE_COLS, seq_len=SEQ_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)



Pobieranie map stacji urządzeń chroniących przebiegi sekwencyjne...
Mapowanie wskaźników z dysku do RAM (Zero realnego Loadingu)...


Kompilacja chronologiczna (Indices): 100%|██████████| 599/599 [00:01<00:00, 437.51it/s]


## 3. Akceleracja Treningu na Tensorach

In [4]:
device = get_device()
print(f"\nKarta Obliczeniowa Sprzęgowo Uruchomiona: {device}")

def train_model(model, title, file_name):
    print(f"\n{'='*50}\nRozpoczynanie uczenia: {title}\n{'='*50}")
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    model = model.to(device)
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        
        bar = tqdm(train_loader, desc=f"Epoka {epoch+1}/{NUM_EPOCHS}")
        for batch_idx, (X_batch, y_batch) in enumerate(bar):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            outputs = model(X_batch)
            loss = criterion(outputs.squeeze(), y_batch.squeeze())
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            if batch_idx % 50 == 0:
                bar.set_postfix({'MSE Loss': f"{running_loss / (batch_idx + 1):.4e}"})
                
        epoch_loss = running_loss / len(train_loader)
        print(f"--> Ukończono Epokę {epoch+1}. Wyliczono Średni błąd kwadratowy (MSE): {epoch_loss:.5e}")
        
    os.makedirs('saved_models', exist_ok=True)
    save_path = f'saved_models/{file_name}.pth'
    torch.save(model.state_dict(), save_path)
    print(f"\n[SUKCES] Parametry Architektury `{title}` zamrożone na dysku: {save_path}")


Karta Obliczeniowa Sprzęgowo Uruchomiona: cuda


In [5]:
INPUT_SIZE = len(train_dataset.feature_cols)

In [ ]:
lstm_net = TimeSeriesLSTM(
    input_size=INPUT_SIZE,
    hidden_size=128,
    num_layers=2,
    output_size=1,
    dropout=0.2
)
train_model(model=lstm_net, title="Recursive Long Short-Term Memory Network", file_name="model_lstm")


Rozpoczynanie uczenia: Recursive Long Short-Term Memory Network


Epoka 1/3: 100%|██████████| 28982/28982 [1:00:32<00:00,  7.98it/s, MSE Loss=1.9644e-03]


--> Ukończono Epokę 1. Wyliczono Średni błąd kwadratowy (MSE): 1.96369e-03


Epoka 2/3: 100%|██████████| 28982/28982 [1:00:33<00:00,  7.98it/s, MSE Loss=1.2276e-03]


--> Ukończono Epokę 2. Wyliczono Średni błąd kwadratowy (MSE): 1.22749e-03


Epoka 3/3: 100%|██████████| 28982/28982 [1:00:52<00:00,  7.94it/s, MSE Loss=1.1302e-03]

--> Ukończono Epokę 3. Wyliczono Średni błąd kwadratowy (MSE): 1.13013e-03

[SUKCES] Parametry Architektury `Recursive Long Short-Term Memory Network` zamrożone na dysku: saved_models/model_lstm.pth


In [6]:
transformer_net = TimeSeriesTransformer(
    input_size=INPUT_SIZE, 
    d_model=128,          
    nhead=4,              
    num_layers=2,         
    output_size=1, 
    dim_feedforward=512,  
    dropout=0.1, 
    max_seq_len=SEQ_LEN
)
train_model(model=transformer_net, title="Attention-based Deep Transformer Encoder", file_name="model_transformer")


Rozpoczynanie uczenia: Attention-based Deep Transformer Encoder


Epoka 1/3: 100%|██████████| 28982/28982 [1:28:45<00:00,  5.44it/s, MSE Loss=2.5044e-03]


--> Ukończono Epokę 1. Wyliczono Średni błąd kwadratowy (MSE): 2.50356e-03


Epoka 2/3: 100%|██████████| 28982/28982 [1:28:51<00:00,  5.44it/s, MSE Loss=1.6697e-03]


--> Ukończono Epokę 2. Wyliczono Średni błąd kwadratowy (MSE): 1.66964e-03


Epoka 3/3: 100%|██████████| 28982/28982 [1:29:28<00:00,  5.40it/s, MSE Loss=1.5325e-03]

--> Ukończono Epokę 3. Wyliczono Średni błąd kwadratowy (MSE): 1.53233e-03

[SUKCES] Parametry Architektury `Attention-based Deep Transformer Encoder` zamrożone na dysku: saved_models/model_transformer.pth
